# Drive_me_crazy pdformer

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np  
from sklearn.metrics import mean_absolute_error, mean_squared_error  

# Run the analysis notebook 
%run drive_me_crazy_tradi.ipynb

In [ ]:

class PropagationDelayAwarePDFormer(nn.Module):
    def __init__(self, input_dim, output_dim=1, d_model=64, nhead=4, num_layers=2, max_propagation_delay=100):
        
        # Base class initialization
        super().__init__()
        
        self.d_model = d_model
        self.max_propagation_delay = max_propagation_delay
        
        # Embedding used to encode discrete propagation delay values
        self.propagation_delay_embedding = nn.Embedding(max_propagation_delay, d_model)
        
        # Linear projection to transform input features to model dimension
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Transformer Encoder Layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model*4,
            batch_first=True  # ensures input shape 
        )
        
        # Stack of Transformer layers
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Ooutput prediction layer
        self.output_layer = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(d_model//2, output_dim)
        )
        
    def forward(self, x, propagation_delays=None):
        """
        x: shape [batch_size, seq_length, input_dim]
        propagation_delays: [batch_size] optional embedding values
        """
        
        batch_size, seq_len, _ = x.shape
        
        # Project raw input features to model dimension
        x_proj = self.input_projection(x)  # [batch, seq, d_model]
        
        # Add propagation delay embeddings only if provided
        if propagation_delays is not None:
            # Clamp delay values within available embedding range
            propagation_delays = torch.clamp(propagation_delays, 0, self.max_propagation_delay-1)
            
            delay_emb = self.propagation_delay_embedding(propagation_delays.long())
            # Broadcasting: add delay embedding to every timestep
            x_proj = x_proj + delay_emb.unsqueeze(1)
        
        # === Causal mask ===
        # Prevents model from "seeing the future" during autoregression
        causal_mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1)
        causal_mask = causal_mask.to(x_proj.device)  
        
        # Transformer forward pass
        transformer_out = self.transformer_encoder(x_proj, mask=causal_mask)
        
        # Prediction based on last timestep (common in forecasting models)
        output = self.output_layer(transformer_out[:, -1, :])
        
        return output

# === Create sequential time-series training data ===
def create_sequential_data(df, sequence_length=24):
    """
    Creates sliding-window sequences for forecasting.
    Returns input sequences, targets, and simulated propagation delays.
    """
    
    features = ['distance_km', 'hour', 'weekday', 'month', 'is_weekend', 'passenger_count']
    
    sequences = []
    targets = []
    propagation_delays = []
    
    for i in range(len(df) - sequence_length):
        seq = df[features].iloc[i:i+sequence_length].values
        target = df['trip_duration_min'].iloc[i+sequence_length]
        
        sequences.append(seq)
        targets.append(target)
        
        # Simulated propagation delay based on last timestep
        avg_delay = min(int(seq[-1, 0] * 2 + seq[-1, 1]), 99)
        propagation_delays.append(avg_delay)
    
    return np.array(sequences), np.array(targets), np.array(propagation_delays)


print("=== PROPAGATION DELAY-AWARE PDFORMER ===")

# Create sequential dataset
X_seq, y_seq, prop_delays = create_sequential_data(df_taxi, sequence_length=24)

# Train/test split
X_train_seq, X_test_seq, y_train_seq, y_test_seq, delays_train, delays_test = train_test_split(
    X_seq, y_seq, prop_delays, test_size=0.2, random_state=42
)

# Convert to tensors
X_train_tensor = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_seq, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_seq, dtype=torch.float32).view(-1, 1)
delays_train_tensor = torch.tensor(delays_train, dtype=torch.float32)
delays_test_tensor = torch.tensor(delays_test, dtype=torch.float32)

# Instantiate model
model_pd = PropagationDelayAwarePDFormer(
    input_dim=X_train_seq.shape[2],
    output_dim=1,
    d_model=64,
    nhead=4,
    num_layers=2
)

# Loss function & optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_pd.parameters(), lr=0.001)

# === Training loop ===
for epoch in range(10):
    model_pd.train()
    epoch_loss = 0
    
    for i in range(0, len(X_train_tensor), 64):
        batch_X = X_train_tensor[i:i+64]
        batch_y = y_train_tensor[i:i+64]
        batch_delays = delays_train_tensor[i:i+64]
        
        optimizer.zero_grad()
        outputs = model_pd(batch_X, batch_delays)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/10], Loss: {epoch_loss/len(X_train_tensor):.4f}")

# === Evaluation ===
model_pd.eval()
with torch.no_grad():
    y_pred_pd = model_pd(X_test_tensor, delays_test_tensor).numpy().flatten()

mae_pd = mean_absolute_error(y_test_seq, y_pred_pd)
rmse_pd = np.sqrt(mean_squared_error(y_test_seq, y_pred_pd))

print("\nPropagation Delay-Aware PDFormer :")
print(f"MAE = {mae_pd:.2f} min")
print(f"RMSE = {rmse_pd:.2f} min")
